In [0]:
# 02_silver_cleaning
from pyspark.sql import functions as F

df_bronze_trips = spark.table("workspace.default.bronze_yellow_taxi")
df_zones = spark.table("workspace.default.bronze_taxi_zone_lookup")

business_cols = [c for c in df_bronze_trips.columns if not c.startswith("_")]
df_bronze_trips = df_bronze_trips.dropDuplicates(business_cols)

valid_zone_ids = [row.LocationID for row in df_zones.select("LocationID").distinct().collect()]

df = df_bronze_trips.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
)
df = df.withColumn(
    "_reject_reason",
    F.when(F.col("fare_amount") <= 0, "invalid_fare")
     .when(F.col("fare_amount") > 500, "fare_too_high")
     .when(F.col("trip_distance") <= 0, "invalid_distance")
     .when(F.col("trip_duration_min") <= 0, "nonpositive_duration")
     .when(F.col("trip_duration_min") > 1440, "duration_too_long")
     .when(~F.col("PULocationID").isin(valid_zone_ids), "unknown_pickup_zone")
     .when(~F.col("DOLocationID").isin(valid_zone_ids), "unknown_dropoff_zone")
     .otherwise(None)
)

df_clean = df.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
df_rejected = df.filter(F.col("_reject_reason").isNotNull())
df_clean = df_clean.withColumn(
    "passenger_count",
    F.when((F.col("passenger_count") <= 0) | (F.col("passenger_count") > 6) | F.col("passenger_count").isNull(), None)
     .otherwise(F.col("passenger_count"))
)

pu_zones = df_zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("PUBorough"), F.col("Zone").alias("PUZone"))
do_zones = df_zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("DOBorough"), F.col("Zone").alias("DOZone"))
df_clean_enriched = df_clean.join(pu_zones, on="PULocationID", how="left").join(do_zones, on="DOLocationID", how="left")

df_clean_enriched.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_yellow_taxi")
df_rejected.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_yellow_taxi_rejected")
print(f"Silver complete. Clean: {df_clean_enriched.count()}, Rejected: {df_rejected.count()}")

Silver complete. Clean: 3911821, Rejected: 179015
